# Phase-Space Adaptive Moving Average Dataset Demo

This demo notebook loads the Phase-Space Adaptive Moving Average Dataset, parses time series history windows, and evaluates a 3-point moving average predictor against a naive last-value baseline.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-4b74fb-self-normalized-phase-space-adaptive-mov/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data = load_data()
print(f"Loaded dataset: {data['datasets'][0]['dataset']} with {len(data['datasets'][0]['examples'])} examples.")

## Configuration

Define tunable parameters for sliding window evaluation.

In [ ]:
# Configurable parameters
WINDOW_SIZE = 10
MA_WINDOW = 3
MAX_SAMPLES = 50

## Processing and Evaluation

Extract time series history and targets from dataset examples, then compare naive last-value forecasting with a 3-point moving average.

In [ ]:
examples = data['datasets'][0]['examples'][:MAX_SAMPLES]

y_true = []
naive_pred = []
ma3_pred = []

for ex in examples:
    inp = json.loads(ex["input"])
    history = inp["history"]
    target = float(ex["output"])
    
    # Naive forecast: last value in history
    n_pred = history[-1]
    # 3-point moving average forecast
    m_pred = np.mean(history[-MA_WINDOW:])
    
    y_true.append(target)
    naive_pred.append(n_pred)
    ma3_pred.append(m_pred)

y_true = np.array(y_true)
naive_pred = np.array(naive_pred)
ma3_pred = np.array(ma3_pred)

naive_mse = np.mean((y_true - naive_pred) ** 2)
ma3_mse = np.mean((y_true - ma3_pred) ** 2)

print(f"Naive MSE: {naive_mse:.4f}")
print(f"3-point MA MSE: {ma3_mse:.4f}")
print(f"3-point MA beats naive: {ma3_mse < naive_mse}")

## Results Visualization

Plot actual targets against naive and 3-point moving average predictions.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(y_true, label="Actual Target", color="black", linewidth=2)
plt.plot(naive_pred, label=f"Naive Forecast (MSE: {naive_mse:.3f})", linestyle="--", color="red")
plt.plot(ma3_pred, label=f"3-Point MA Forecast (MSE: {ma3_mse:.3f})", linestyle="-", color="blue")
plt.title("Forecasting Comparison on Phase-Space Dataset")
plt.xlabel("Sample Index")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()